###  SANDBOX: PIPELINE CALL CENTER

Este Notebook actúa como un entorno de experimentación y laboratorio de pruebas (Sandbox) para validar las funciones, transformaciones y lógica de negocio antes de migrarlas a los scripts definitivos en la carpeta `src/`.


### FASE 0: CONFIGURACIÓN DE INFRAESTRUCTURA Y ENTORNO

En esta sección se gestiona el aislamiento del entorno de ejecución, la importación de librerías base y la validación de los requisitos mínimos del sistema.

#### Objetivos:
* Importar las librerías oficiales (`pandas`, `os`).
* Definir y validar la accesibilidad de las rutas relativas hacia las fuentes de datos.
Markdown

In [24]:
# Importamos las librerías necesarias
import os
import pandas as pd
import re

In [5]:
# Definición de rutas relativas desde la carpeta 'notebooks/'
# El uso de '../' es obligatorio para salir de 'notebooks/' y poder entrar a 'data/'
ruta_interacciones = '../data/raw/Data_Robust_Call_Center_Log_Interacciones.xlsx'
ruta_maestro = '../data/raw/Data_Robust_Call_Center_Maestro_Clientes.xlsx'

In [7]:
# Verificación física de archivos (Devuelve True o False)
existe_interacciones = os.path.exists(ruta_interacciones)
existe_maestro = os.path.exists(ruta_maestro)

print("==== VERIFICACIÓN DE INFRAESTRUCTURA ====")
print(f"¿Existe el archivo de Interacciones?: {existe_interacciones}")
print(f"¿Existe el archivo de Maestro de Clientes?: {existe_maestro}")

==== VERIFICACIÓN DE INFRAESTRUCTURA ====
¿Existe el archivo de Interacciones?: True
¿Existe el archivo de Maestro de Clientes?: True


In [8]:
# Compuerta de seguridad analítica
if not existe_interacciones or not existe_maestro:
    raise FileNotFoundError("Error crítico: No se puede iniciar el pipeline porque faltan archivos en data/raw/")

### FASE 1: EXTRACCIÓN (RAW DATA)
Proceso encargado de realizar la lectura de los componentes en bruto ubicados en el directorio `data/raw/`.

* Validar la existencia física de los archivos mediante el sistema operativo.
* Cargar las pestañas específicas de los libros de Excel (`Log_Interacciones` y `Maestro_Clientes`) a la memoria RAM como DataFrames de Pandas.
* Normalizar técnicamente las cabeceras (remoción de espacios en blanco invisibles y sustitución de espacios intermedios por guiones bajos) para asegurar la compatibilidad con operaciones avanzadas.

In [9]:
# 1. Extracción e Ingesta del Log de Interacciones
df_interacciones_raw = pd.read_excel(ruta_interacciones, sheet_name='Log_Interacciones', engine='openpyxl')

In [10]:
# Algoritmo de normalización de columnas (Elimina espacios extremos y reemplaza espacios internos por '_')
df_interacciones_raw.columns = [str(c).strip().replace(' ', '_') for c in df_interacciones_raw.columns]

In [16]:
# Verificación del tipo de dato por campo y muestra de las primeras filas
print("=== INSPECCIÓN DE ESTRUCTURA: LOG INTERACCIONES ===")
df_interacciones_raw.info()
display(df_interacciones_raw.head(5))

=== INSPECCIÓN DE ESTRUCTURA: LOG INTERACCIONES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Interaction_ID    100 non-null    object
 1   Customer_ID       100 non-null    object
 2   Call_Date         100 non-null    object
 3   Duration_Seconds  100 non-null    int64 
 4   Status            100 non-null    object
 5   Queue             100 non-null    object
dtypes: int64(1), object(5)
memory usage: 4.8+ KB


,Interaction_ID,Customer_ID,Call_Date,Duration_Seconds,Status,Queue
0,INT_1000,CLI_118,2026-06-05,33,Answered,Soporte
1,INT_1001,CLI_119,2026-06-03,801,Answered,Soporte
2,INT_1002,CLI_131,2026-06-06,365,Answered,Ventas
3,INT_1003,CLI_106,2026-06-03,393,Answered,Reclamos
4,INT_1004,CLI_151,2026-06-03,57,Answered,Retenciones


In [11]:
# 2. Extracción e Ingesta del Maestro de Clientes
df_clientes_raw = pd.read_excel(ruta_maestro, sheet_name='Maestro_Clientes', engine='openpyxl')

In [12]:
# Algoritmo de normalización de columnas
df_clientes_raw.columns = [str(c).strip().replace(' ', '_') for c in df_clientes_raw.columns]

In [17]:
# Verificación del tipo de dato por campo y muestra de las primeras filas
print("\n=== INSPECCIÓN DE ESTRUCTURA: MAESTRO CLIENTES ===")
df_clientes_raw.info()
display(df_clientes_raw.head(5))


=== INSPECCIÓN DE ESTRUCTURA: MAESTRO CLIENTES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Customer_ID          60 non-null     object
 1   Customer_Name        60 non-null     object
 2   Segment              60 non-null     object
 3   Phone_Number         60 non-null     int64 
 4   Has_Active_Campaign  60 non-null     object
dtypes: int64(1), object(4)
memory usage: 2.5+ KB


,Customer_ID,Customer_Name,Segment,Phone_Number,Has_Active_Campaign
0,CLI_100,Guillermo Suero,Gold,999411855,NO
1,CLI_101,Andree Roldan,Regular,996267180,SI
2,CLI_102,Xiomara Gomez,Regular,998807313,SI
3,CLI_103,Carlos Mendoza,Regular,993704238,NO
4,CLI_104,Ana Torres,Premium,993630708,SI


In [18]:
# Verificación de la cantidad de filas y columnas para ambos datasets
print("=== REPORTE DE EXTRACCIÓN CONSOLIDADO ===")
print(f"Log Interacciones -> Filas: {len(df_interacciones_raw)} | Columnas: {len(df_interacciones_raw.columns)}")
print(f"Maestro Clientes  -> Filas: {len(df_clientes_raw)} | Columnas: {len(df_clientes_raw.columns)}\n")

=== REPORTE DE EXTRACCIÓN CONSOLIDADO ===
Log Interacciones -> Filas: 100 | Columnas: 6
Maestro Clientes  -> Filas: 60 | Columnas: 5



### FASE 2: TRANSFORMACIÓN (REGLAS DE NEGOCIO)
Etapa crítica del pipeline donde se manipula la estructura y consistencia de los datos mediante lógica matemática y reglas operativas corporativas.

* Identificar y tratar valores nulos, duplicados o anomalías en los registros.
* Ejecutar el cruce de información (Join/Merge) entre la tabla transaccional y el maestro de clientes mediante la clave primaria `Customer_ID`.
* Realizar conversiones de tipos de datos (formateo de campos de fecha y duraciones).
* Calcular métricas de negocio y KPIs solicitados por el área usuaria.

In [20]:
# Creamos las copias de trabajo oficiales eliminando el sufijo '_raw'
# Esto protege los datos originales extraídos en la Fase 1 ante modificaciones
df_interacciones = df_interacciones_raw.copy()
df_clientes = df_clientes_raw.copy()

In [ ]:
# =====================================================================
# FASE 2.1: TRATAMIENTO DE ANOMALÍAS (REGLA DE NEGOCIO 1)
# =====================================================================

# 1. Filtrar llamadas abandonadas que tienen una duración mayor a 0 segundos
falsos_abandonos = df_interacciones[
    (df_interacciones['Status'] == 'Abandoned') & 
    (df_interacciones['Duration_Seconds'] > 0)
]

print("=== DIAGNÓSTICO DE CALIDAD DE DATOS ===")
print(f"Cantidad de llamadas 'Abandoned' con duración errónea (> 0s): {len(falsos_abandonos)}")

if len(falsos_abandonos) > 0:
    print("\nMuestra de registros anómalos detectados:")
    display(falsos_abandonos.head(5))
else:
    print("\nNo se detectaron anomalías en la relación Status/Duration.")

print("-" * 50)

=== DIAGNÓSTICO DE CALIDAD DE DATOS ===
Cantidad de llamadas 'Abandoned' con duración errónea (> 0s): 4

Muestra de registros anómalos detectados:


,Interaction_ID,Customer_ID,Call_Date,Duration_Seconds,Status,Queue
18,INT_1018,CLI_130,2026-06-01,28,Abandoned,Soporte
21,INT_1021,CLI_134,2026-06-04,8,Abandoned,Retenciones
85,INT_1085,CLI_106,2026-06-01,7,Abandoned,Ventas
97,INT_1097,CLI_129,2026-06-06,7,Abandoned,Soporte


--------------------------------------------------


In [ ]:
# 2. Forzar Duration_Seconds = 0 utilizando .loc
# Sintaxis: df.loc[condición_filas, columna_a_modificar] = nuevo_valor
df_interacciones.loc[df_interacciones['Status'] == 'Abandoned', 'Duration_Seconds'] = 0

print("=== VERIFICACIÓN POST-TRATAMIENTO ===")
# Volvemos a evaluar la condición para asegurar que el contador baje a 0
anomalias_remanentes = df_interacciones[
    (df_interacciones['Status'] == 'Abandoned') & 
    (df_interacciones['Duration_Seconds'] > 0)
]
print(f"Cantidad de anomalías después de la limpieza: {len(anomalias_remanentes)}")
print(f"AHT promedio global actual de la muestra: {df_interacciones['Duration_Seconds'].mean():.2f} segundos")

=== VERIFICACIÓN POST-TRATAMIENTO ===
Cantidad de anomalías después de la limpieza: 0
AHT promedio global actual de la muestra: 194.92 segundos


In [ ]:
# ==========================================================================
# FASE 2.2: SANITIZACIÓN DE CONTACTOS MEDIANTE EXPRESIONES REGULARES (REGEX)
# ==========================================================================
# 1. Aseguramos que la columna sea interpretada como string limpio de espacios en los extremos
df_clientes['Phone_Number'] = df_clientes['Phone_Number'].astype(str).str.strip()

print("=== MUESTRA ANTES DE LA SANITIZACIÓN ===")
display(df_clientes[['Customer_ID', 'Phone_Number']].head(5))
print("-" * 50)

=== MUESTRA ANTES DE LA SANITIZACIÓN ===


,Customer_ID,Phone_Number
0,CLI_100,999411855
1,CLI_101,996267180
2,CLI_102,998807313
3,CLI_103,993704238
4,CLI_104,993630708


--------------------------------------------------


In [ ]:
# 2. Definición del algoritmo regex para limpiar números de teléfono
# \D es el patrón Regex que busca "cualquier caracter que NO sea un dígito numérico"
def limpiar_telefono(telefono_raw):
    # 1. Eliminar todo lo que no sea número
    telefono_limpio = re.sub(r'\D', '', telefono_raw)
    
    # 2. Validación de regla de negocio de longitud
    # Evaluamos si el teléfono resultante tiene una longitud menor a 7 dígitos (anomalía extrema)
    if len(telefono_limpio) < 7:
        return "REVISAR_TELEFONO"
    
    return telefono_limpio

In [ ]:
# 3. Aplicación del algoritmo fila por fila usando una función lambda
df_clientes['Phone_Number_Clean'] = df_clientes['Phone_Number'].apply(limpiar_telefono)

print("=== MUESTRA POST-SANITIZACIÓN ===")
display(df_clientes[['Customer_ID', 'Phone_Number', 'Phone_Number_Clean']].head(5))

=== MUESTRA POST-SANITIZACIÓN ===


,Customer_ID,Phone_Number,Phone_Number_Clean
0,CLI_100,999411855,999411855
1,CLI_101,996267180,996267180
2,CLI_102,998807313,998807313
3,CLI_103,993704238,993704238
4,CLI_104,993630708,993630708


In [ ]:
# 4. Control de calidad analítico: Contar cuántos registros necesitan auditoría manual
telefonos_erroneos = df_clientes[df_clientes['Phone_Number_Clean'] == 'REVISAR_TELEFONO']
print(f"\nCantidad de registros marcados para revisión analítica: {len(telefonos_erroneos)}")


Cantidad de registros marcados para revisión analítica: 0


In [ ]:
# ==========================================================================
# FASE 2.3: ENRIQUECIMIENTO DE DATOS (MERGE / JOIN)
# ==========================================================================
# 1. Ejecución del left join
# pd.merge toma la tabla izquierda, la derecha, la columna llave y el tipo de unión
df_consolidado = pd.merge(
    df_interacciones, 
    df_clientes[['Customer_ID', 'Customer_Name', 'Segment', 'Phone_Number_Clean', 'Has_Active_Campaign']], 
    on='Customer_ID', 
    how='left'
)

print("=== VERIFICACIÓN DEL CRUCE DE DATOS ===")
print(f"Dimensiones de la tabla final -> Filas: {df_consolidado.shape[0]} | Columnas: {df_consolidado.shape[1]}")

In [ ]:
# 2. Control de calidad analítico: Verificar si hubo llamadas de clientes que no existen en el maestro
clientes_no_encontrados = df_consolidado['Customer_Name'].isna().sum()
print(f"Cantidad de interacciones sin match en el maestro: {clientes_no_encontrados}")

print("\nMuestra de las primeras 5 filas del DataFrame Consolidado:")
display(df_consolidado.head(5))

### FASE 3: CARGA (PROCESSED DATA)
Fase final de transferencia de datos encargada de dar salida a la información procesada y optimizada.

* Validar que la estructura final del DataFrame consolidado cumpla con los tipos de datos requeridos.
* Exportar el set de datos transformado a un archivo de almacenamiento físico en la carpeta `data/processed/`.
* Asegurar que el formato de salida sea óptimo para su consumo posterior en herramientas de analítica o tableros de control.

### FASE 4: ORQUESTACIÓN Y LOGS (SIMULACIÓN)
Espacio dedicado a emular el comportamiento del script central `main.py` bajo un entorno controlado.

* Estructurar el flujo secuencial de las fases anteriores mediante un bloque único de ejecución.
* Probar el control de excepciones (`try-except`) ante posibles fallos de infraestructura o inconsistencias de datos.